# Project: Reinforcement Learning Trees (RLT) — Pipeline


# **1. Business Understanding**

This project follows the CRISP-DM methodology and aims to **evaluate the performance of Reinforcement Learning Trees (RLT)** compared to other machine-learning models (Random Forests, Gradient Boosting, SVM, LASSO/Logistic Regression) across **ten different datasets** from the UCI Machine Learning Repository.

The study replicates the experimental setup described in the research article summarised in the file `/mnt/data/DESCRIPTION.docx`.  
Each dataset represents a real-world predictive task (classification or regression), such as housing value prediction, disease diagnosis, wine quality estimation, concrete strength prediction, and more.

### **Main Business Objective**
To determine **whether RLT provides superior predictive accuracy and variable-selection capability** under high-dimensional conditions (p = 500 features) by:

- sampling 150 observations for training,
- expanding the feature space to 500 variables using controlled noise injection,
- comparing RLT's performance with widely used baseline models.

Understanding how RLT behaves across diverse datasets helps organizations or researchers:

- select the right model for noisy, high-dimensional problems,
- improve prediction accuracy,
- better understand which variables drive outcomes,
- reduce decision-making risk in fields such as healthcare, engineering, finance, and manufacturing.

The final goal is **a fair, standardized comparison** of model robustness across multiple tasks and domains.




# **2. Data Understanding**

The experiment uses **ten datasets** from the UCI Machine Learning Repository, each representing a different predictive problem:

1. Boston Housing — regression  
2. Parkinsons — classification  
3. Sonar — classification  
4. White Wine — regression  
5. Red Wine — regression  
6. Parkinson Oxford — regression  
7. Ozone — regression  
8. Concrete Strength — regression  
9. Breast Cancer — classification  
10. Auto MPG — regression  

Each dataset differs in:

- number of samples,  
- number of original features,  
- presence of numeric and categorical variables,  
- level of noise,  
- scale (units) and distribution shapes,  
- missing values and inconsistencies.

### **General Observations**
- Some datasets contain **missing values** (“?” in UCI format).
- Many features have **different scales**, requiring standardization.
- Several datasets mix **numerical** and **categorical** variables.
- Targets differ: some are continuous (regression), others categorical (classification).
- Some datasets have **few features**, so extra **noise variables** must be generated to reach **p = 500**, as required by the RLT experimental setup.
- Because datasets vary in size, a fixed **150-sample training set** is created for all of them to ensure fair comparison.

### **Purpose of Data Understanding**
The goal is to:

- assess structural differences between datasets,
- identify necessary cleaning steps,
- recognize variable types,
- verify target column correctness,
- understand whether the dataset is for classification or regression,
- prepare each dataset in a **consistent, standardized way** for the experimental pipeline.

This ensures that every dataset can be processed identically and evaluated fairly in the modeling phase.



## Cell 1 — Data Cleaning (universal function)
This cell defines a general-purpose cleaning function to handle the 10 UCI datasets:
- Replace "?" with NaN
- Drop rows with missing target
- Convert numeric-like strings to numeric
- Impute numeric missing values with median
- Impute categorical missing values with mode
- Drop obvious ID columns if present
- Return cleaned dataframe


In [8]:
# Cell 1: Data Cleaning
import pandas as pd
import numpy as np

DESCRIPTION_PATH = "/mnt/data/DESCRIPTION.docx"   # reference to your uploaded description file

def clean_dataset(df, target_col=None, drop_id_cols=True):
    """
    Generic cleaning for UCI datasets:
    - replace '?' with NaN
    - drop rows with missing target (if provided)
    - convert numeric-like object columns to numeric where possible
    - impute numeric cols with median, categorical with mode
    - optionally drop common id columns (encounter_id, patient_nbr)
    Returns cleaned DataFrame.
    """
    df = df.copy()
    df = df.replace("?", np.nan)

    # Drop common id columns (diabetes dataset example)
    if drop_id_cols:
        for col in ["encounter_id", "patient_nbr", "id", "Id", "ID"]:
            if col in df.columns:
                df = df.drop(columns=[col])

    # If target supplied, drop rows where target is missing
    if target_col is not None and target_col in df.columns:
        df = df.dropna(subset=[target_col])

    # Attempt to convert object cols that are numeric-like
    obj_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in obj_cols:
        # skip obvious non-categorical text columns (short heuristic)
        try:
            df[col] = pd.to_numeric(df[col])
        except:
            pass

    # Recompute column types
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in df.columns if c not in numeric_cols]

    # Impute numeric with median
    for c in numeric_cols:
        if df[c].isnull().any():
            median = df[c].median()
            df[c] = df[c].fillna(median)

    # Impute categorical with mode (or "Missing" if mode fails)
    for c in categorical_cols:
        if df[c].isnull().any():
            try:
                mode = df[c].mode()[0]
            except:
                mode = "Missing"
            df[c] = df[c].fillna(mode)

    return df

# Quick usage example:
# df = pd.read_csv("data/boston.csv")
# df_clean = clean_dataset(df, target_col="MEDV")


##  — Data Preparation (standardize, train/test split, expand to p=500)
This cell implements the per-dataset preparation used in the article:
- standardize continuous variables (mean 0, var 1) using training data stats
- sample 150 observations (without replacement) as training set; remainder as test set
- add artificial noise features until total features p = 500 (signal-to-noise ratio ~1:2)
- returns X_train, y_train, X_test, y_test (pandas DataFrames/Series)


In [11]:
# Cell 2: Data Preparation
from sklearn.preprocessing import StandardScaler

def prepare_dataset_for_experiment(df, target_col, task_type="classification", n_train=150, p_target=500, seed=42):
    """
    Prepare dataset according to the article:
    - df: cleaned dataframe
    - target_col: name of label column
    - task_type: "classification" or "regression"
    - n_train: number of train samples to sample
    - p_target: expand features until p_target total columns
    Returns: X_train, y_train, X_test, y_test
    """
    np.random.seed(seed)
    df = df.copy()

    # 1) Separate X/y
    X = df.drop(columns=[target_col])
    y = df[target_col].reset_index(drop=True)

    # 2) Train/test split: sample 150 rows for training (without replacement)
    if len(df) <= n_train:
        raise ValueError(f"Dataset has {len(df)} rows which is <= n_train ({n_train}). Choose smaller n_train.")
    train_idx = df.sample(n=n_train, random_state=seed).index
    test_idx = df.index.difference(train_idx)

    X_train = X.loc[train_idx].reset_index(drop=True)
    y_train = y.loc[train_idx].reset_index(drop=True)
    X_test = X.loc[test_idx].reset_index(drop=True)
    y_test = y.loc[test_idx].reset_index(drop=True)

    # 3) Standardize continuous features using training stats
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler = StandardScaler()
    if numeric_cols:
        X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

    # 4) If there are object/categorical columns, one-hot encode them (same for train/test)
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    if cat_cols:
        # Use pandas get_dummies with drop_first to reduce dimension
        X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
        X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)
        # Align columns (fill missing dummies with 0)
        X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

    # 5) Add noise features until p_target achieved (constructed from randomly sampled existing numeric feature)
    # If no numeric columns we generate pure noise features
    base_numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    while X_train.shape[1] < p_target:
        if base_numeric_cols:
            base = np.random.choice(base_numeric_cols)
            # signal-to-noise ratio: we create noise with larger variance (approx SNR 1:2)
            # new = base + eps where eps ~ Normal(0, 2*std(base))
            std_base = max(1e-6, X_train[base].std())  # in case std is 0
            eps_train = np.random.normal(loc=0.0, scale=2*std_base, size=len(X_train))
            eps_test = np.random.normal(loc=0.0, scale=2*std_base, size=len(X_test))
            new_train = X_train[base].values + eps_train
            new_test = X_test[base].values + eps_test
        else:
            # fallback: pure gaussian noise
            new_train = np.random.normal(size=len(X_train))
            new_test = np.random.normal(size=len(X_test))

        colname = f"noise_{X_train.shape[1]}"
        X_train[colname] = new_train
        X_test[colname] = new_test

    # Final alignment (just in case)
    X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

    return X_train, y_train, X_test, y_test

# Quick usage example:
# Xtr, ytr, Xte, yte = prepare_dataset_for_experiment(df_clean, target_col="Class", task_type="classification")
